# Gradient descent 

By now you have all the conceptual knowledge of how deep learning actually works. 
In this lecture, we will put it into action by creating the actual learning process: 
**the training loop**.

Each time backpropagation is computed and W and b are updated with new values, 
the model makes a new prediction. This loop continues until the loss no longer 
improves, and that is when the learning is complete.

Thus, the training loop is where the training happens, and it is driven by a algorithm called **gradient descent**:


$$
\theta_{t+1}
= \theta_t
-  n \, \nabla_\theta L
$$

Where 
- $\theta$ (theta) represents all our parameters, i.e. W and b
- $\eta$ (eta) is the learning rate. A small number controlling the step size for each parameter update
- $\nabla_\theta L$ is the gradient of the loss, i.e. the W.grad and b.grad we calculated in the last lecture

As you can see, the algorithm is not as scary as it might first seem. 
In code, it looks something like this:

- W_new = W_old - learning_rate * W.grad
- b_new = b_old - learning_rate * b.grad

What this code essentially does is move our parameters in the opposite 
direction of the gradient, scaled by the learning rate.


## The training loop

So now you have learned all the five steps of deep learning! 

- Prediction
- Loss calculation 
- Gradient calculation
- Parameter update 
- Gradient reset 

Now lets put it to a loop where the five steps are repeated for multiple epochs. Do you remember what epochs and batches was? 

- Batches are the chunks of a dataset the modell trains on for every iteration
- Epoch is when the full dataset have gone trough the modell once 

You will see the following code snipets in this lecture, and it is important that you now what they do

- torch.no_grad()
    - Tells PyTorch NOT to track parameter updates
- .grad.zero_()
    - Resets gradients for each iteration 

## Create data for this lecture

First, lets create our data again from the previous lecture. 

In [6]:
import torch

# Our batch have 10 samples 
N = 10
# Each sample has 1 input feature and 1 output value 
D_in = 1
D_out = 1

# Create input data
X = torch.randn(N, D_in)

# Create our "ground truth", i.e. the weight and bias
# that will result in a y_hat with the same value as
# our actual y
true_W = torch.tensor([[2.0]]) # true W is 2.0
true_b = torch.tensor(1.0) # true b is 1.0

##!! In real-world applications the model never sees the true W or b.
# We only have input data and the corresponding target labels.
# The model must learn the underlying W and b by comparing its
# predictions to the ground truth and adjusting its parameters
# to minimize the error.

# Create target labels + noise
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1

# Input and ground truth label should be same size
print("X shape:", X.shape)
print("y_true shape:", y_true.shape)

# Initialize the model's parameters with random values.
# W has shape (D_in, D_out) so it can be multiplied 
# with X of shape (N, D_in). 
# b has shape (1,) and will be broadcast across all samples.
# Setting requires_grad=True makes PyTorch track these 
# tensors as learnable parameters whose gradients will be 
# computed during backprop.
W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# What you see is the models initial prediction which
# of course will be completly wrong! 
print(f"Initial W: {W}\n")
print(f"Initial b: {b}")

# ----- 3. FORWARD PASS -----
# Our manual linear model: y_hat = X @ W + b
y_hat = X @ W + b

# Print the first 5 predicted values.
# These will be random at first because the model has not been trained yet.
# NOTICE THAT grad_fn IS ACTIVE!! It will say SliceBackward
# since our last operation to y_hat is indexing the five first rows
print(f"Prediction y_hat (only first 5 rows):\n{y_hat[:5]}\n")

# Print the first 5 true labels so we can compare.
print(f"True labels y_true (only first 5 rows): \n{y_true[:5]}")


# Calculate error
error = y_hat - y_true
# Square error
squared_error = error ** 2
# Take the average
loss = squared_error.mean()

print(f"Loss: {loss}")

# Compute the gradients
loss.backward()

print(f"Gradient for W: \n {W.grad}")
print(f"\nGradient for b: {b.grad}")

X shape: torch.Size([10, 1])
y_true shape: torch.Size([10, 1])
Initial W: tensor([[0.3367]], requires_grad=True)

Initial b: tensor([-1.2250], requires_grad=True)
Prediction y_hat (only first 5 rows):
tensor([[-0.7087],
        [-1.1405],
        [-1.3899],
        [-1.7513],
        [-1.0028]], grad_fn=<SliceBackward0>)

True labels y_true (only first 5 rows): 
tensor([[ 4.1293],
        [ 1.5122],
        [-0.0870],
        [-2.1478],
        [ 2.2530]])
Loss: 8.1783447265625
Gradient for W: 
 tensor([[-3.7686]])

Gradient for b: tensor([-4.5248])


In [7]:
# -----------------------------
# Hyperparameters
# -----------------------------

# How big steps we take in the direction opposite the gradient.
# Too big = unstable training. Too small = very slow learning.
learning_rate = 0.01   

# How many times we loop through the entire training process.
# Each loop is one full forward + backward + update cycle.
epochs = 300           


# -----------------------------
# Parameter initialization
# -----------------------------

# We create W and b as trainable parameters.
# torch.randn initializes them with random values drawn from a normal distribution.
# requires_grad=True tells PyTorch to track operations on them so gradients can be computed.
W = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)


# -----------------------------
# Training Loop
# -----------------------------

for epoch in range(epochs):

    # -------------------------
    # Forward pass
    # -------------------------
    # Compute model predictions: y_hat = XW + b
    # This is the linear model we are training.
    y_hat = X @ W + b

    # Compute the loss (Mean Squared Error).
    # We measure how far the predictions are from the true values.
    loss = torch.mean((y_hat - y_true)**2)


    # -------------------------
    # Backward pass
    # -------------------------
    # Computes d(loss)/d(W) and d(loss)/d(b)
    # PyTorch builds the computation graph automatically and backpropagates through it.
    loss.backward()


    # -------------------------
    # Parameter update
    # -------------------------
    # We update W and b manually using gradient descent:
    # new_value = old_value - learning_rate * gradient
    #
    # torch.no_grad() is used because we do NOT want PyTorch to track these operations.
    # Updating parameters is not part of the computational graph.
    with torch.no_grad():
        W -= learning_rate * W.grad   # Move W opposite the gradient direction
        b -= learning_rate * b.grad   # Same for b


    # -------------------------
    # Reset gradients
    # -------------------------
    # PyTorch accumulates gradients by default.
    # If we don't zero them out, gradients from previous iterations will stack up.
    W.grad.zero_()
    b.grad.zero_()


    # -------------------------
    # Track the learning process
    # -------------------------

    if epoch % 20 == 0:
        print(f"Epoch {epoch:02d}: Loss{loss.item():.4f}, W={W.item():.3f}, b={b.item():.3f}")

print(f"\nFinal parameters: W={W.item():.3f}, b={b.item():.3f}")
print(f"True parameters: W=2.0, b=1.0")

Epoch 00: Loss18.6154, W=-1.412, b=-1.170
Epoch 20: Loss7.7475, W=-0.181, b=-0.426
Epoch 40: Loss3.2264, W=0.609, b=0.062
Epoch 60: Loss1.3451, W=1.115, b=0.380
Epoch 80: Loss0.5619, W=1.440, b=0.589
Epoch 100: Loss0.2359, W=1.649, b=0.726
Epoch 120: Loss0.1000, W=1.783, b=0.815
Epoch 140: Loss0.0434, W=1.868, b=0.874
Epoch 160: Loss0.0198, W=1.923, b=0.912
Epoch 180: Loss0.0100, W=1.958, b=0.937
Epoch 200: Loss0.0059, W=1.981, b=0.954
Epoch 220: Loss0.0041, W=1.995, b=0.965
Epoch 240: Loss0.0034, W=2.005, b=0.972
Epoch 260: Loss0.0031, W=2.011, b=0.977
Epoch 280: Loss0.0030, W=2.014, b=0.980

Final parameters: W=2.017, b=0.982
True parameters: W=2.0, b=1.0


Notice how the loss decreases with every epoch, and how W and b move closer to 
their true values. This is training happening in real time! After 300 epochs, 
the final parameters are very close to the true ones, and the model converges well.


## Summary

We have now successfully implemented the entire gradient descent algorithm 
from scratch and built a machine that learns. Take a moment to review the 
lectures once more, and feel free to ask an LLM to clarify anything that 
still feels fuzzy. Then complete the exercises below before moving on to 
the next lecture, where we will learn how to perform this process much 
faster and more easily using torch.nn.Module!


# Exercise 9 - Tensor Shapes and Broadcasting Logic

You are given the following tensors:

X has shape (100, 1)
W has shape (1, 1)
b has shape (1,)

1. Without running any code, explain:
   - What the shape of y_hat = X @ W + b will be.
   - Why broadcasting allows b to be added to X @ W.
   - What would happen if W had shape (1,) instead of (1,1).

2. Describe a scenario where a shape mismatch would occur, and explain
   how you would fix it.

3. Explain why understanding tensor shapes is essential for building
   neural networks.


# Exercise 10 - Reasoning About Autograd and the Computation Graph

Consider the following operations:

z = X @ W
y_hat = z + b
loss = mean((y_hat - y_true)**2)

1. Draw (conceptually, not visually) the computation graph:
   - List each node and what operation it represents.
   - Identify which nodes require gradients and which do not.

2. Explain in your own words how PyTorch knows how to compute:
   d(loss)/dW and d(loss)/db

3. Describe what would happen if:
   - requires_grad=False for W
   - You forgot to call .backward()
   - You forgot to zero the gradients


# Exercise 11 - Understandning the Meaning of the Gradient

Suppose W.grad is positive and large after backpropagation.

1. Explain what this tells you about:
   - The direction the loss increases when W increases.
   - How gradient descent will update W.

2. Describe a situation where:
   - The gradient is extremely small.
   - The gradient is extremely large.

3. For each situation, explain:
   - What it means for the learning process.
   - How you might adjust the learning rate.


# Exercise 12 - Reasoning About the Training Loop

A training loop performs the following steps:

1. Forward pass
2. Compute loss
3. Backward pass
4. Update parameters
5. Zero gradients

For each step:

1. Explain WHY it must happen in that order.
2. Describe what would break if the order were changed.
3. Explain why the update step must be inside torch.no_grad().
4. Explain why gradients accumulate by default and why this is useful.


# Exercise 5 - Convergence, Stability and Learning Dynamics

Imagine you train a simple linear model for 300 epochs.

You observe the following:

- The loss decreases at first but then plateaus.
- W and b move closer to the true values but never reach them exactly.
- The loss curve is smooth and stable.

1. Explain what "convergence" means in this context.
2. Give three possible reasons why the model does not reach the exact
   true parameters.
3. Explain how each of the following would affect convergence:
   - A larger learning rate
   - A smaller learning rate
   - Noisy data
   - Poor parameter initialization

4. Describe how you would determine whether the model is:
   - Underfitting
   - Overfitting
   - Learning correctly
